# Loading the libraries    

In [12]:
import io, os
import json
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from dotenv import load_dotenv

In [13]:
load_dotenv()

True

# Loading OpenAI Model

In [81]:
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
OPEN_AI_MODEL = os.getenv("OPEN_AI_MODEL")

In [82]:
open_ai_model = ChatOpenAI(model_name=OPEN_AI_MODEL, openai_api_key=OPENAI_API_KEY, temperature=0, max_tokens=1000)

# Compare Reviews based on Quality

In [83]:
COMPARISON_SYSTEM_PROMPT="""You are an expert software engineer evaluating two code review reports for the same pull request.

Compare Review A and Review B and determine which review provides higher quality feedback.

Evaluate based on:
- Number and importance of issues identified
- Correctness of findings
- Severity accuracy
- Actionability of recommendations
- Presence of incorrect or misleading findings

Do not consider wording or length. Focus on the quality and usefulness of the review.

Return ONLY the following JSON:

{
  "better_review": "A | B | Equivalent",
  "score_difference": "<0-10 indicating how much better the selected review is>",
  "A_score": "<0-10>",
  "B_score": "<0-10>",
}"""

In [84]:
def compare_reviews(base_review, candidate_review):
    response = open_ai_model.invoke([
        SystemMessage(content=COMPARISON_SYSTEM_PROMPT),
        HumanMessage(content=f"Review A: {base_review}\nReview B: {candidate_review}")
    ])
    return response.content

# Diagnosis

In [85]:
ai_review1="Suggested fix: Remove the `return None` statement or ensure it is placed correctly"
ai_review2="Description: Unreachable statements after the return statementSuggested fix: Remove the unreachable return statement `return None`."

In [86]:
print(compare_reviews(ai_review1, ai_review2))

{
  "better_review": "B",
  "score_difference": 2,
  "A_score": 5,
  "B_score": 7
}


# Measure Comparative Quality

In [87]:
def load_repo_data(repo_path):
    with open(repo_path, "r") as f:
        return f.read()

In [88]:
def load_code_changes(json_content):
   return json.loads(json_content)

In [89]:
def extract_base_candidate_elements(base_repo_path, faulty_repo_path):
    repo_data=load_repo_data(base_repo_path)
    base_elements=load_code_changes(repo_data)
    faulty_repo_data=load_repo_data(faulty_repo_path)
    fault_elements=load_code_changes(faulty_repo_data)
    return base_elements, fault_elements

In [24]:
import time

In [90]:
def process_comparison_result(result):
    try:
        item=json.loads(result)
        if not item:
            return "Comparison is not possible"
        else:
            return f"{item["better_review"]}, {item["score_difference"]}, {item["A_score"]}, {item["B_score"]}"     
    except Exception as e:
        print(f"Error occurred:{result}")

In [94]:
def calculate_relative_quality(base_elements, candidate_elements):
    for index in range(len(base_elements)):
        try:
            time.sleep(3)
            base_review_comment=base_elements[index]["review_comment"]
            candidate_review_comment=candidate_elements[index]["review_comment"]
            result=compare_reviews(base_review_comment, candidate_review_comment)
            #print(f"Index: {index}, Raw: {result}")    
            print(f"Index: {index}, {process_comparison_result(result)}")
        except Exception as e:
            print(index, e)

In [124]:
repo_name="pandas"
base_repo_path=f"dataset/reviewed-ext/{repo_name}.json"
candidate_repo_path=f"dataset/faulty/{repo_name}-message-corrupted-fault.json"
base_elements, fault_elements=extract_base_candidate_elements(base_repo_path, candidate_repo_path)
print(len(base_elements), len(fault_elements))

14 14


In [ ]:
calculate_relative_quality(base_elements, fault_elements)

Index: 0, A, 3, 8, 5
Index: 1, A, 6, 9, 3
Index: 2, A, 5, 8, 3
Index: 3, A, 5, 8, 3
Index: 4, A, 5, 8, 3
Index: 5, A, 5, 8, 3
Index: 6, A, 5, 9, 4
Index: 7, B, 8, 2, 10
Index: 8, A, 3, 7, 4
Index: 9, A, 5, 8, 3
Index: 10, A, 6, 9, 3
Index: 11, A, 5, 8, 3
Index: 12, A, 5, 8, 3
Index: 13, B, 6, 2, 8


: 